# PLS Regression on UV–Vis Spectral Data (SPZ files)

Samples 1–25 | Target Concentration from Excel

In [ ]:

import numpy as np
import pandas as pd
import zipfile
import os
import matplotlib.pyplot as plt

from sklearn.cross_decomposition import PLSRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import r2_score, mean_squared_error


## Function to read SPZ files

In [ ]:

def read_spz(file_path):
    with zipfile.ZipFile(file_path, 'r') as z:
        inner_file = z.namelist()[0]
        with z.open(inner_file) as f:
            df = pd.read_csv(f, sep=None, engine='python')
    wavelength = df.iloc[:, 0].values
    absorbance = df.iloc[:, 1].values
    return absorbance, wavelength


## Load spectral data

In [ ]:

spectra = []
sample_ids = []

data_dir = "/mnt/data"

for i in range(1, 26):
    file_name = f"PW_{i}.spz"
    file_path = os.path.join(data_dir, file_name)
    absorbance, wavelength = read_spz(file_path)
    spectra.append(absorbance)
    sample_ids.append(f"PW_{i}")

X = np.array(spectra)
print("Spectral matrix shape:", X.shape)


## Load target concentration from Excel

In [ ]:

y_df = pd.read_excel("concentration.xlsx")
y_df = y_df.set_index("sample_id").loc[sample_ids]
y = y_df["conc_ppm"].values
print("Target vector shape:", y.shape)


## Preprocessing

In [ ]:

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


## PLS Regression

In [ ]:

n_components = 5
pls = PLSRegression(n_components=n_components)

y_pred = cross_val_predict(pls, X_scaled, y, cv=5)

r2 = r2_score(y, y_pred)
rmse = np.sqrt(mean_squared_error(y, y_pred))

print("R²:", r2)
print("RMSE:", rmse)


## Actual vs Predicted Plot

In [ ]:

plt.figure(figsize=(6,6))
plt.scatter(y, y_pred)
plt.plot([y.min(), y.max()], [y.min(), y.max()], '--')
plt.xlabel("Actual Concentration (ppm)")
plt.ylabel("Predicted Concentration (ppm)")
plt.title("PLS Regression: Actual vs Predicted")
plt.show()


## Train final model

In [ ]:

pls.fit(X_scaled, y)
